In [10]:
import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from pathlib import Path


In [18]:
LAG_FEATURES = ["PrevLapTime", "PrevLapTime2", "RollingMean3", "RollingMean5",
                 "RollingStd5", "PaceTrend"]

df = pd.read_csv("../../Data/Processed/Master/featured_dataset.csv")

model_features= Path("D:/Automotive/Race Strategy Decision Support System (RSDSS)/Race-Strategy-Decision-Support-System-RSDSS-/src/Models/model_features.pkl")

TARGET = "LapTime_Seconds"  # adjust if your target column has a different name
model_features = joblib.load(model_features)
df = df[df["ValidLap"]].copy()
train_df = df[df["Year"].isin([2022, 2023, 2024])].copy()
test_df = df[df["Year"] == 2025].copy()

y_train = train_df[TARGET]
y_test = test_df[TARGET]

train_df = pd.get_dummies(train_df)
test_df = pd.get_dummies(test_df)

train_df = train_df.reindex(columns=model_features, fill_value=False)

test_df = test_df.reindex(columns=model_features, fill_value=False)

Option A: drop lag features, force the model to lean on TireAge/context

In [19]:
features_a = [f for f in model_features if f not in LAG_FEATURES]

X_train_a = train_df[features_a].dropna()
X_test_a = test_df[features_a].dropna()

model_a = GradientBoostingRegressor(n_estimators=100, max_depth=3,
                                     learning_rate=0.1, random_state=42)
model_a.fit(X_train_a, y_train)
pred_a = model_a.predict(X_test_a)



print("=== Option A: lag features dropped ===")
print(f"MAE: {mean_absolute_error(y_test, pred_a):.3f}")
print(f"R2:  {r2_score(y_test, pred_a):.3f}")
imp_a = sorted(zip(features_a, model_a.feature_importances_), key=lambda x: -x[1])
for name, val in imp_a[:10]:
    print(f"  {name:25s} {val:.4f}")
print()

ValueError: Found input variables with inconsistent numbers of samples: [70415, 71024]

Option B: keep lag features, but predict the DELTA vs previous lap instead of the raw lap time. This keeps useful lag info without letting the model just copy PrevLapTime as its answer -- the target itself no longer correlates almost 1:1 with a single input feature.

In [ ]:
train_df["LapTimeDelta"] = train_df[TARGET] - train_df["PrevLapTime"]
test_df["LapTimeDelta"] = test_df[TARGET] - test_df["PrevLapTime"]

# drop rows where PrevLapTime was NaN (first lap of each stint/session)
train_df_b = train_df.dropna(subset=["LapTimeDelta"])
test_df_b = test_df.dropna(subset=["LapTimeDelta"])

X_train_b = train_df_b[model_features]
y_train_b = train_df_b["LapTimeDelta"]
X_test_b = test_df_b[model_features]
y_test_b = test_df_b["LapTimeDelta"]

model_b = GradientBoostingRegressor(n_estimators=100, max_depth=3,
                                     learning_rate=0.1, random_state=42)
model_b.fit(X_train_b, y_train_b)
pred_delta_b = model_b.predict(X_test_b)

# reconstruct actual lap time prediction: PrevLapTime + predicted delta
pred_b_laptime = test_df_b["PrevLapTime"].values + pred_delta_b

print("=== Option B: predict delta vs PrevLapTime ===")
print(f"MAE (delta):     {mean_absolute_error(y_test_b, pred_delta_b):.3f}")
print(f"MAE (laptime):   {mean_absolute_error(test_df_b[TARGET], pred_b_laptime):.3f}")
print(f"R2  (laptime):   {r2_score(test_df_b[TARGET], pred_b_laptime):.3f}")
imp_b = sorted(zip(model_features, model_b.feature_importances_), key=lambda x: -x[1])
for name, val in imp_b[:10]:
    print(f"  {name:25s} {val:.4f}")
print()

print("Compare MAE/R2 between A and B against your current model's baseline")
print("MAE/R2 on the same 2025 test set. Pick whichever keeps accuracy close")
print("to baseline while giving TireAge/TireLifePercentage real importance")
print("(ideally double digits %, not ~1-2%).")
print()
print("Once you pick one, save it:")
print("  joblib.dump(model_a, 'gradient_boosting_model_v2.pkl')")
print("  joblib.dump(features_a, 'model_features_v2.pkl')")
print("(swap in model_b/features_b if you go with Option B -- and update")
print(" DegradationModel.predict_lap_time in race_simulator.py to add")
print(" PrevLapTime + predicted delta if you choose Option B, since the")
print(" model now outputs a delta, not a raw lap time.)")